# CSV 통합

In [100]:
import numpy as np
import pandas as pd
from IPython.display import display
import warnings
import platform
import matplotlib.pyplot as plt

# 출력 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams[
        'font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


## 사용 함수

In [101]:
# 컬럼 정보 간단 표현
def check_basic_info(df, df_name, exclude_cols=None):
    
    print(f"\n{'='*80}")
    print(f"{df_name}의 컬럼 정보 / 결측치 확인 정보 요약")
    print(f"{'='*80}\n")


    # 제외할 컬럼 반영
    df_copied = df.copy()
    if exclude_cols:
        df_copied = df_copied.drop(columns=exclude_cols, errors='ignore')
    
    # 1. 전체 요약
    overview_df = pd.DataFrame({
        '항목': ['행 개수', '열 개수', '중복 행 개수'],
        '값': [df_copied.shape[0], df_copied.shape[1], df_copied.duplicated().sum()]
    })
    
    # 2. 컬럼별 요약
    summary_df = pd.DataFrame({
        '데이터타입': df_copied.dtypes.astype(str),
        '행 개수': df_copied.count(),
        '행 비율(%)': (df_copied.count() / len(df) * 100).round(2),
        '결측치 개수': df_copied.isnull().sum(),
        '결측치 비율(%)': (df_copied.isnull().sum() / len(df) * 100).round(2),
        '고유값 개수': df_copied.nunique(dropna=True)
    })
    
    # 3. 보기 좋게 정렬
    summary_df = summary_df.sort_values(
        by=['결측치 개수', '고유값 개수'],
        ascending=[False, False]
    )
    
    print("[전체 요약]")
    display(overview_df)
    
    print("[컬럼별 요약]")
    display(summary_df)

    print("[테이블 요약]")
    display(df.head())

In [102]:
# 중복값 확인
def check_id_duplicates(df, col_name, df_name):
    print(f"\n{'='*80}")
    print(f"{df_name}의 값 중복 확인")
    print(f"{'='*80}")
    
    print("전체 행 수:", len(df))
    print(f"{col_name} 고유 개수:", df[col_name].nunique())
    print(f"중복 {col_name} 개수:", df[col_name].duplicated().sum())

In [103]:
# 컬럼 분포 확인
def check_category_summary(df, df_name, col_name):
    
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 범주 확인")
    print(f"{'='*80}")
    
    summary_df = df[col_name].value_counts(dropna=False).reset_index()
    summary_df.columns = [col_name, '개수']
    summary_df['비율(%)'] = (summary_df['개수'] / len(df) * 100).round(2)
    
    display(summary_df.head(10))

# 테이블 전처리

In [104]:
# ============================================================
# 1. 원본 데이터 로드
# ============================================================
df_portfolio = pd.read_csv("data/portfolio.csv", index_col=0)
df_profile = pd.read_csv("data/profile.csv", index_col=0)
df_transcript = pd.read_csv("data/transcript.csv", index_col=0)
df_menu = pd.read_csv("data/starbucks_menu_260112.csv", index_col=0)

## df_portfolio 전처리

In [105]:
# ============================================================
# 확인 및 복제
# ============================================================
print("[portfolio]")
df_portfolio_copy = df_portfolio.copy()
check_basic_info(df_portfolio_copy, "portfolio")

[portfolio]

portfolio의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,10
1,열 개수,6
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
id,str,10,100.0,0,0.0,10
reward,int64,10,100.0,0,0.0,5
difficulty,int64,10,100.0,0,0.0,5
duration,int64,10,100.0,0,0.0,5
channels,str,10,100.0,0,0.0,4
offer_type,str,10,100.0,0,0.0,3


[테이블 요약]


,reward,channels,difficulty,duration,offer_type,id
0,10,"['email', 'mobile', 'social']",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,10,"['web', 'email', 'mobile', 'social']",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,0,"['web', 'email', 'mobile']",0,4,informational,3f207df678b143eea3cee63160fa8bed
3,5,"['web', 'email', 'mobile']",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9
4,5,"['web', 'email']",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7


In [106]:
# ============================================================
# 컬럼명 정리
# ============================================================
df_portfolio_clean = (
    df_portfolio_copy
    .rename(columns={
    'id': 'offer_id',
    'reward': 'offer_reward'
    })
)
display(df_portfolio_clean.head())

,offer_reward,channels,difficulty,duration,offer_type,offer_id
0,10,"['email', 'mobile', 'social']",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,10,"['web', 'email', 'mobile', 'social']",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,0,"['web', 'email', 'mobile']",0,4,informational,3f207df678b143eea3cee63160fa8bed
3,5,"['web', 'email', 'mobile']",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9
4,5,"['web', 'email']",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7


In [107]:
# ============================================================
# channels 문자열을 리스트로 변환
# ============================================================
channel_list = ['web', 'email', 'mobile', 'social']


for ch in channel_list:
    df_portfolio_clean[ch] = (
        df_portfolio_clean['channels']
        .str
        .contains(ch)
        .astype(int)
    )

display(df_portfolio_clean.head())

,offer_reward,channels,difficulty,duration,offer_type,offer_id,web,email,mobile,social
0,10,"['email', 'mobile', 'social']",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd,0,1,1,1
1,10,"['web', 'email', 'mobile', 'social']",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0,1,1,1,1
2,0,"['web', 'email', 'mobile']",0,4,informational,3f207df678b143eea3cee63160fa8bed,1,1,1,0
3,5,"['web', 'email', 'mobile']",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9,1,1,1,0
4,5,"['web', 'email']",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7,1,1,0,0


## df_portfolio 전처리 최종 확인

In [108]:
check_basic_info(df_portfolio_clean, "portfolio_clean")
check_id_duplicates(df_portfolio_clean, 'offer_id', 'portfolio_clean')


portfolio_clean의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,10
1,열 개수,10
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
offer_id,str,10,100.0,0,0.0,10
offer_reward,int64,10,100.0,0,0.0,5
difficulty,int64,10,100.0,0,0.0,5
duration,int64,10,100.0,0,0.0,5
channels,str,10,100.0,0,0.0,4
offer_type,str,10,100.0,0,0.0,3
web,int64,10,100.0,0,0.0,2
mobile,int64,10,100.0,0,0.0,2
social,int64,10,100.0,0,0.0,2
email,int64,10,100.0,0,0.0,1


[테이블 요약]


,offer_reward,channels,difficulty,duration,offer_type,offer_id,web,email,mobile,social
0,10,"['email', 'mobile', 'social']",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd,0,1,1,1
1,10,"['web', 'email', 'mobile', 'social']",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0,1,1,1,1
2,0,"['web', 'email', 'mobile']",0,4,informational,3f207df678b143eea3cee63160fa8bed,1,1,1,0
3,5,"['web', 'email', 'mobile']",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9,1,1,1,0
4,5,"['web', 'email']",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7,1,1,0,0



portfolio_clean의 값 중복 확인
전체 행 수: 10
offer_id 고유 개수: 10
중복 offer_id 개수: 0


## df_profile 전처리

In [109]:
# ============================================================
# 확인 및 복제
# ============================================================
print("\n[profile]")
df_profile_copy = df_profile.copy()
check_basic_info(df_profile_copy, "profile")


[profile]

profile의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,17000
1,열 개수,5
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
income,float64,14825,87.21,2175,12.79,91
gender,str,14825,87.21,2175,12.79,3
id,str,17000,100.00,0,0.00,17000
became_member_on,int64,17000,100.00,0,0.00,1716
age,int64,17000,100.00,0,0.00,85


[테이블 요약]


,gender,age,id,became_member_on,income
0,NaN,118,68be06ca386d4c31939f3a4f0e3dd783,20170212,NaN
1,F,55,0610b486422d4921ae7d2bf64640c50b,20170715,112000.0
2,NaN,118,38fe809add3b4fcf9315a9694bb96ff5,20180712,NaN
3,F,75,78afa995795e4d85b5d9ceeca43f5fef,20170509,100000.0
4,NaN,118,a03223e636434f42ac4c3df47e8bac43,20170804,NaN


In [110]:
# ============================================================
# 컬럼명 정리
# ============================================================
df_profile_copy = (
    df_profile_copy
    .rename(columns={
    'id': 'customer_id'
}))

In [111]:
# ============================================================
# 가입일 날짜형 변환
# ============================================================
df_profile_copy['became_member_on'] = pd.to_datetime(
    df_profile_copy['became_member_on']
    .astype(str),
    format='%Y%m%d'
)

In [112]:
# ============================================================
# 가입일 파생 컬럼
# ============================================================
# 파생컬럼 필요시 사용
# # 가입 년도
# df_profile_copy['member_year'] = (
#     df_profile_copy['became_member_on']
#     .dt
#     .year
# )

# # 가입 월
# df_profile_copy['member_month'] = (
#     df_profile_copy['became_member_on'].
#     dt
#     .month
# )

# # 가입 분기
# df_profile_clean['member_quarter'] = (
#     df_profile_clean['became_member_on']
#     .dt
#     .quarter
# )

In [113]:
# ============================================================
# age 이상치 처리
# ============================================================
# 드롭하기 보단 NaN으로 처리
# 미기재 고객일 가능성도 배제할순 없음

df_profile_copy['age'] = (
    df_profile_copy['age'].
    replace(118, np.nan)
)

print("age 결측 개수:", df_profile_copy['age'].isna().sum())
print("gender 결측 개수:", df_profile_copy['gender'].isna().sum())
print("income 결측 개수:", df_profile_copy['income'].isna().sum())

# 결측 여부 플래그 생성
df_profile_copy['age_missing'] = df_profile_copy['age'].isna().astype(int)
df_profile_copy['gender_missing'] = df_profile_copy['gender'].isna().astype(int)
df_profile_copy['income_missing'] = df_profile_copy['income'].isna().astype(int)

age 결측 개수: 2175
gender 결측 개수: 2175
income 결측 개수: 2175


In [114]:
# ============================================================
# 컬럼 순서 정리
# ============================================================
df_profile_clean = df_profile_copy[
    [
        'customer_id',
        'gender', 'age', 'income',
        'gender_missing', 'age_missing', 'income_missing'
    ]
]

## df_profile 전처리 최종확인

In [115]:
check_basic_info(df_profile_clean, "profile_clean")
check_id_duplicates(df_profile_clean, 'customer_id', 'profile_clean')


profile_clean의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,17000
1,열 개수,7
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
income,float64,14825,87.21,2175,12.79,91
age,float64,14825,87.21,2175,12.79,84
gender,str,14825,87.21,2175,12.79,3
customer_id,str,17000,100.00,0,0.00,17000
gender_missing,int64,17000,100.00,0,0.00,2
age_missing,int64,17000,100.00,0,0.00,2
income_missing,int64,17000,100.00,0,0.00,2


[테이블 요약]


,customer_id,gender,age,income,gender_missing,age_missing,income_missing
0,68be06ca386d4c31939f3a4f0e3dd783,NaN,NaN,NaN,1,1,1
1,0610b486422d4921ae7d2bf64640c50b,F,55.0,112000.0,0,0,0
2,38fe809add3b4fcf9315a9694bb96ff5,NaN,NaN,NaN,1,1,1
3,78afa995795e4d85b5d9ceeca43f5fef,F,75.0,100000.0,0,0,0
4,a03223e636434f42ac4c3df47e8bac43,NaN,NaN,NaN,1,1,1



profile_clean의 값 중복 확인
전체 행 수: 17000
customer_id 고유 개수: 17000
중복 customer_id 개수: 0


## df_transcript 전처리

In [116]:
# ============================================================
# 확인 및 복제
# ============================================================
print("[df_transcript]")
df_transcript_copy = df_transcript.copy()
check_basic_info(df_transcript_copy, "profile")
check_category_summary(df_transcript_copy, "transcript", "event")

[df_transcript]

profile의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,306534
1,열 개수,4
2,중복 행 개수,397


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
person,str,306534,100.0,0,0.0,17000
value,str,306534,100.0,0,0.0,5121
time,int64,306534,100.0,0,0.0,120
event,str,306534,100.0,0,0.0,4


[테이블 요약]


,person,event,value,time
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0
1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0
2,e2127556f4f64592b11af22de27a7932,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0
4,68617ca6246f4fbc85e91a2a49552598,offer received,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0



transcript의 event 범주 확인


,event,개수,비율(%)
0,transaction,138953,45.33
1,offer received,76277,24.88
2,offer viewed,57725,18.83
3,offer completed,33579,10.95


In [117]:
# ============================================================
# 컬럼명 정리
# ============================================================
df_transcript_copy = (
    df_transcript_copy
    .rename(columns={
    'person': 'customer_id'
}))

In [118]:
# ============================================================
# value 컬럼 파싱
# ============================================================
import ast
# ast란?

# ============================================================
# value 컬럼 딕셔너리로 형변환
# value 컬럼은 문자열처럼 보이지만 사실 딕셔너리 형태
# 먼저 컬럼을 딕셔너리의 형태로 변환한다.
# ============================================================
df_transcript_copy['value'] = (
    df_transcript_copy['value']
    .apply(ast.literal_eval))

print(df_transcript_copy['value'].apply(type).value_counts())

value
<class 'dict'>    306534
Name: count, dtype: int64


In [119]:
# ============================================================
# value 컬럼 키 구조 확인
# ============================================================
event_keys = sorted({
    key
    for d in df_transcript_copy['value']
    for key in d.keys()
})

print("value 컬럼 전체 key 종류:")
print(event_keys)

value 컬럼 전체 key 종류:
['amount', 'offer id', 'offer_id', 'reward']


In [120]:
# ============================================================
# value 컬럼(dict) 분리 및 주요 파생 컬럼 생성
# event별로 value 안에 들어 있는 값을 별도 컬럼으로 분리
# 'offer id'와 'offer_id'는 하나의 offer_id 컬럼으로 통합
# ============================================================
value_df = pd.DataFrame(df_transcript_copy['value'].tolist())
df_transcript_copy = pd.concat([df_transcript_copy, value_df], axis=1)

# offer id / offer_id 컬럼명을 하나로 통합
df_transcript_copy['offer_id'] = df_transcript_copy['offer id'].fillna(df_transcript_copy['offer_id'])

df_transcript_copy = (
    df_transcript_copy
    .rename(columns={
    'reward': 'event_reward'
    })
)

# 주요 컬럼 확인
display(
    df_transcript_copy[
        [
            'customer_id', 
            'event', 
            'time', 
            'offer_id', 
            'amount', 
            'event_reward']]
        .head()
)

,customer_id,event,time,offer_id,amount,event_reward
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN


In [121]:
# ============================================================
# event별 컬럼 결측 개수 확인
# ============================================================
print("="*80)
print("이벤트별 결측 확인")
print("="*80)

print(df_transcript_copy[['event', 'offer_id', 'amount', 'event_reward']].isna().groupby(df_transcript_copy['event']).sum())

이벤트별 결측 확인
                 event  offer_id  amount  event_reward
event                                                 
offer completed      0         0   33579             0
offer received       0         0   76277         76277
offer viewed         0         0   57725         57725
transaction          0    138953       0        138953


In [122]:
df_transcript_clean = df_transcript_copy[
    [
        'customer_id',
        'event',
        'time',
        'offer_id',
        'amount',
        'event_reward',
        'value'
    ]
]

## df_transcript 전처리 최종확인

In [123]:
check_basic_info(df_transcript_clean, "transcript_clean", exclude_cols = 'value')
check_id_duplicates(df_transcript_clean, 'customer_id', 'transcript_clean(customer_id)')
check_id_duplicates(df_transcript_clean, 'offer_id', 'transcript_clean(offer_id)')


transcript_clean의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,306534
1,열 개수,6
2,중복 행 개수,397


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
event_reward,float64,33579,10.95,272955,89.05,4
amount,float64,138953,45.33,167581,54.67,5103
offer_id,str,167581,54.67,138953,45.33,10
customer_id,str,306534,100.00,0,0.00,17000
time,int64,306534,100.00,0,0.00,120
event,str,306534,100.00,0,0.00,4


[테이블 요약]


,customer_id,event,time,offer_id,amount,event_reward,value
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'}
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'}
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,{'offer id': '2906b810c7d4411798c6938adc9daaa5'}
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'}
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'}



transcript_clean(customer_id)의 값 중복 확인
전체 행 수: 306534
customer_id 고유 개수: 17000
중복 customer_id 개수: 289534

transcript_clean(offer_id)의 값 중복 확인
전체 행 수: 306534
offer_id 고유 개수: 10
중복 offer_id 개수: 306523


# 코드 통합

## transcript과 profile 매칭 확인

In [124]:
# 조인키: customer_id 
print("transcript customer_id 고유 개수:", df_transcript_clean['customer_id'].nunique())
print("profile customer_id 고유 개수:", df_profile_clean['customer_id'].nunique())

unmatched_customers = set(df_transcript_clean['customer_id']) - set(df_profile_clean['customer_id'])
print("profile에 없는 transcript 고객 수:", len(unmatched_customers))

transcript customer_id 고유 개수: 17000
profile customer_id 고유 개수: 17000
profile에 없는 transcript 고객 수: 0


## transcript과 portfolio 매칭 확인

In [125]:
# 조인키: offer_id 
transcript_offer_ids = set(df_transcript_clean['offer_id'].dropna())
portfolio_offer_ids = set(df_portfolio_clean['offer_id'])

print("transcript 내 offer_id 개수:", len(transcript_offer_ids))
print("portfolio 내 offer_id 개수:", len(portfolio_offer_ids))
print("portfolio에 없는 transcript offer_id 수:", len(transcript_offer_ids - portfolio_offer_ids))

transcript 내 offer_id 개수: 10
portfolio 내 offer_id 개수: 10
portfolio에 없는 transcript offer_id 수: 0


## transcript + profile

In [126]:
# ============================================================
# transcript + profile 컬럼 조인
# ============================================================
df_transcript_profile = df_transcript_clean.merge(
    df_profile_clean,
    on='customer_id',
    how='left'
)

df_transcript_profile = df_transcript_profile[
    [
        'customer_id', 'event', 'time',
        'offer_id', 'amount', 'event_reward',
        'gender', 'age', 'income',
        'gender_missing', 'age_missing', 'income_missing',
        'value'
    ]
]


In [127]:
print("transcript 원본 행 수:", len(df_transcript_clean))
print("transcript + profile 행 수:", len(df_transcript_profile))
print("행 수 차이:", len(df_transcript_profile) - len(df_transcript_clean))

check_basic_info(df_transcript_profile, "transcript + profile", exclude_cols = 'value')
check_id_duplicates(df_transcript_profile, 'customer_id', 'transcript + profile')

transcript 원본 행 수: 306534
transcript + profile 행 수: 306534
행 수 차이: 0

transcript + profile의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,306534
1,열 개수,12
2,중복 행 개수,397


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
event_reward,float64,33579,10.95,272955,89.05,4
amount,float64,138953,45.33,167581,54.67,5103
offer_id,str,167581,54.67,138953,45.33,10
income,float64,272762,88.98,33772,11.02,91
age,float64,272762,88.98,33772,11.02,84
gender,str,272762,88.98,33772,11.02,3
customer_id,str,306534,100.00,0,0.00,17000
time,int64,306534,100.00,0,0.00,120
event,str,306534,100.00,0,0.00,4
gender_missing,int64,306534,100.00,0,0.00,2


[테이블 요약]


,customer_id,event,time,offer_id,amount,event_reward,gender,age,income,gender_missing,age_missing,income_missing,value
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,F,75.0,100000.0,0,0,0,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'}
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,NaN,NaN,NaN,1,1,1,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'}
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,M,68.0,70000.0,0,0,0,{'offer id': '2906b810c7d4411798c6938adc9daaa5'}
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,NaN,NaN,NaN,1,1,1,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'}
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,NaN,NaN,NaN,1,1,1,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'}



transcript + profile의 값 중복 확인
전체 행 수: 306534
customer_id 고유 개수: 17000
중복 customer_id 개수: 289534


## transcript + portfolio

In [128]:
# ============================================================
# transcript + portfolio 컬럼 조인
# ============================================================
df_transcript_portfolio = df_transcript_clean.merge(
    df_portfolio_clean,
    on='offer_id',
    how='left'
)

df_transcript_portfolio = df_transcript_portfolio[
    [
        'customer_id', 'event', 'time',
        'offer_id', 'amount', 'event_reward',
        'offer_type', 'offer_reward', 'difficulty', 'duration', 'channels',
        'web', 'email', 'mobile', 'social',
        'value'
    ]
]


In [129]:
print("transcript 원본 행 수:", len(df_transcript_clean))
print("transcript + portfolio 행 수:", len(df_transcript_portfolio))
print("행 수 차이:", len(df_transcript_portfolio) - len(df_transcript_clean))

check_basic_info(df_transcript_portfolio, "transcript + portfolio", exclude_cols = 'value')
check_id_duplicates(df_transcript_portfolio, 'offer_id', 'transcript + portfolio')

transcript 원본 행 수: 306534
transcript + portfolio 행 수: 306534
행 수 차이: 0

transcript + portfolio의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,306534
1,열 개수,15
2,중복 행 개수,397


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
event_reward,float64,33579,10.95,272955,89.05,4
amount,float64,138953,45.33,167581,54.67,5103
offer_id,str,167581,54.67,138953,45.33,10
offer_reward,float64,167581,54.67,138953,45.33,5
difficulty,float64,167581,54.67,138953,45.33,5
duration,float64,167581,54.67,138953,45.33,5
channels,str,167581,54.67,138953,45.33,4
offer_type,str,167581,54.67,138953,45.33,3
web,float64,167581,54.67,138953,45.33,2
mobile,float64,167581,54.67,138953,45.33,2


[테이블 요약]


,customer_id,event,time,offer_id,amount,event_reward,offer_type,offer_reward,difficulty,duration,channels,web,email,mobile,social,value
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,bogo,5.0,5.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'}
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,discount,5.0,20.0,10.0,"['web', 'email']",1.0,1.0,0.0,0.0,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'}
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,discount,2.0,10.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,{'offer id': '2906b810c7d4411798c6938adc9daaa5'}
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,discount,2.0,10.0,10.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'}
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,bogo,10.0,10.0,5.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'}



transcript + portfolio의 값 중복 확인
전체 행 수: 306534
offer_id 고유 개수: 10
중복 offer_id 개수: 306523


## transcript + portfolio + profile

In [130]:
# ============================================================
# transcript + portfolio + profile 컬럼 조인
# ============================================================
df_transcript_portfolio_profile = (
    df_transcript_clean
    .merge(df_portfolio_clean, on='offer_id', how='left')
    .merge(df_profile_clean, on='customer_id', how='left')
)

df_transcript_portfolio_profile = df_transcript_portfolio_profile[
    [
        'customer_id', 'event', 'time',
        'offer_id', 'offer_type',
        'amount', 'event_reward', 'offer_reward',
        'difficulty', 'duration', 'channels',
        'gender', 'age', 'income',
        'gender_missing', 'age_missing', 'income_missing',
        'value'
    ]
]

In [131]:
print("transcript 원본 행 수:", len(df_transcript_clean))
print("전체 통합 행 수:", len(df_transcript_portfolio_profile))
print("행 수 차이:", len(df_transcript_portfolio_profile) - len(df_transcript_clean))

check_basic_info(df_transcript_portfolio_profile, "transcript + portfolio + profile", exclude_cols = 'value')
check_id_duplicates(df_transcript_portfolio_profile, 'customer_id', 'transcript + portfolio + profile (customer_id)')
check_id_duplicates(df_transcript_portfolio_profile, 'offer_id', 'transcript + portfolio + profile (offer_id)')

transcript 원본 행 수: 306534
전체 통합 행 수: 306534
행 수 차이: 0

transcript + portfolio + profile의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,306534
1,열 개수,17
2,중복 행 개수,397


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
event_reward,float64,33579,10.95,272955,89.05,4
amount,float64,138953,45.33,167581,54.67,5103
offer_id,str,167581,54.67,138953,45.33,10
offer_reward,float64,167581,54.67,138953,45.33,5
difficulty,float64,167581,54.67,138953,45.33,5
duration,float64,167581,54.67,138953,45.33,5
channels,str,167581,54.67,138953,45.33,4
offer_type,str,167581,54.67,138953,45.33,3
income,float64,272762,88.98,33772,11.02,91
age,float64,272762,88.98,33772,11.02,84


[테이블 요약]


,customer_id,event,time,offer_id,offer_type,amount,event_reward,offer_reward,difficulty,duration,channels,gender,age,income,gender_missing,age_missing,income_missing,value
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,bogo,NaN,NaN,5.0,5.0,7.0,"['web', 'email', 'mobile']",F,75.0,100000.0,0,0,0,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'}
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,discount,NaN,NaN,5.0,20.0,10.0,"['web', 'email']",NaN,NaN,NaN,1,1,1,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'}
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,discount,NaN,NaN,2.0,10.0,7.0,"['web', 'email', 'mobile']",M,68.0,70000.0,0,0,0,{'offer id': '2906b810c7d4411798c6938adc9daaa5'}
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,discount,NaN,NaN,2.0,10.0,10.0,"['web', 'email', 'mobile', 'social']",NaN,NaN,NaN,1,1,1,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'}
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,bogo,NaN,NaN,10.0,10.0,5.0,"['web', 'email', 'mobile', 'social']",NaN,NaN,NaN,1,1,1,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'}



transcript + portfolio + profile (customer_id)의 값 중복 확인
전체 행 수: 306534
customer_id 고유 개수: 17000
중복 customer_id 개수: 289534

transcript + portfolio + profile (offer_id)의 값 중복 확인
전체 행 수: 306534
offer_id 고유 개수: 10
중복 offer_id 개수: 306523


## 통합 csv 파일 보고서


### 오퍼 성과 분석 → transcript_portfolio
1. 행수 증가 없음
2. customer_id 중복 존재
    - transcript는 이벤트 로그 테이블로 한 고객이 여러 번 등장하는 게 정상
    - 예시:\
    offer received/offer viewed/transaction/offer completed 를 여러 번 하면 그만큼 행이 생긴다.
3. gender, age, income 결측값
    - 제거 안함, 필요시 제거

### 고객 특성 분석 → transcript_profile
1. 행수 증가 없음
2. offer_id 중복 존재
    - 오퍼는 10종류 이고 그 오퍼가 여러 고객에게 여러 번 등장,\
    로그 테이블에서 offer_id가 계속 반복되는 게 정상

## 세그먼트 / 종합 분석 → transcript_portfolio_profile
1. 행수 증가 없음
2. customer_id / offer_id 불일치가 이미 없었음
3. 결측 패턴도 자연스러움
    - offer_id, offer_type, offer_reward, difficulty, duration, channels\
    transaction은 원래 특정 offer와 직접 연결되지 않는 행이 많음
    - gender, age, income\
    이건 원래 profile의 결측이 반영, 원본 데이터 특성
4. 중복 확인 결과\
한 고객은 여러 이벤트를 가진다\
같은 오퍼가 여러 고객, 여러 시점에 반복 등장함

# 저장

In [ ]:
# ============================================================
# 통합 파일 저장
# ============================================================
df_transcript_profile.to_csv("transcript_profile.csv", index=False)
df_transcript_portfolio.to_csv("transcript_portfolio.csv", index=False)
df_transcript_portfolio_profile.to_csv("transcript_portfolio_profile.csv", index=False)

print("통합 csv 3종 저장 완료")

통합 csv 3종 저장 완료
